# Lab 3 Worksheet: What Would It Cost?

**Time:** about 25 minutes, alongside `Lab3_Cloud_Deployment_Comparison.md`

The steering committee will ask a simple question: *should we pay a cloud provider to run the language model, or buy our
own server?* This worksheet helps you answer it with numbers.

A cloud LLM (large language model) is billed per token: every question costs a small amount, so the bill grows with use.
A self-hosted one is mostly a fixed cost: the server, its power, and the people who run it, whether you ask it ten
questions a day or ten thousand. It's like the choice between taking taxis and buying a car: taxis are cheaper if you
travel a little; the car wins if you travel a lot.

Which is cheaper depends almost entirely on **volume**, so the useful output of this worksheet isn't one number. It's the
volume at which the answer flips: the **break-even point**.

**Every price in this notebook is a placeholder.** Cloud prices change often and differ by model, region and contract, and
hardware quotations depend on the supplier. Replace each value marked `EXAMPLE` with a current figure from the provider's
price list or the Fund's procurement team before you use the result for anything.

### Words you'll meet in this worksheet

| Word | Plain meaning |
|---|---|
| **token** | the unit LLMs read, write and are billed in; roughly 4 characters of English |
| **input / output tokens** | tokens sent to the model (prompt, passages, question) / tokens it writes back (the answer) |
| **price per million tokens** | how cloud providers quote prices; input and output usually have different prices |
| **fixed cost** | a cost that doesn't change with volume (a server, a salary) |
| **variable cost** | a cost that grows with volume (cloud tokens) |
| **depreciation** | spreading the cost of equipment over the years it's used |
| **FTE** | full-time equivalent: 0.25 FTE = a quarter of one person's working time |
| **peak load** | the busiest minutes; servers must be sized for these, not the average |
| **break-even** | the volume at which two options cost the same |

## Where this worksheet fits

**What you've done so far.** In Lab 1 you built the handbook assistant with RAG, and in Lab 2 you turned it into an
internal API and measured its latency: a few seconds per answer, almost all of it the LLM. On Day 7 you saw that Swahili
uses more tokens than English for the same meaning, and that cloud LLMs are billed per token.

**What you'll do now.** Alongside the written comparison in `Lab3_Cloud_Deployment_Comparison.md`, you'll put numbers on
the choice between paying a cloud provider per token and buying a GPU server. You'll estimate the tokens in one RAG
question, set out the assumptions, work out a month's cost for each option, and find the volume at which the cheaper
option changes.

**Why it matters.** A steering committee decides with a budget in front of it. A clear cost model, with every assumption
visible and replaceable, turns "the cloud is cheaper" or "our own server is cheaper" from an opinion into something that
can be checked and updated when prices change.

**What happens next.** This completes Day 9. On Day 10 you'll deploy a service, monitor it, and assemble the governance
pack, where cost and data-protection arguments like these belong.

### Theory: fixed costs, variable costs and break-even

Every cost is one of two kinds:

- a **fixed cost** stays the same whatever the volume: a server's purchase price (spread over its life), its power, a
  share of an engineer's time;
- a **variable cost** grows with each unit of use: cloud tokens, billed per question.

Draw total cost against volume and the two options look like this:

```
cost │                ╱  cloud: pay per question
     │              ╱
     │ ───────────╳───────────  self-hosted: mostly fixed
     │          ╱ ↑
     │        ╱   break-even
     │      ╱
     │    ╱
     └──────────────────────── questions per day
```

At low volume, paying per use is cheaper: you don't pay for a server that sits idle. At high volume, the fixed option wins,
because its cost is shared across more questions. The **break-even point** is where the lines cross. It's more useful to a
committee than any single monthly figure, because it says *how much use would change the answer*.

## Imports

Just the standard tools: `math` (for rounding up the number of servers), NumPy, pandas and matplotlib for the chart.

In [ ]:
import math

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

> **Where we are.** The tools are loaded. A cloud bill is tokens times price, so start by counting the tokens in one
> question.

## 1. How many tokens does one question use?

Every RAG (retrieval-augmented generation) answer sends the system prompt, four retrieved passages and the question, and
gets back a short answer. Use the real prompts from Lab 1 if you can. Here's an estimate from their typical sizes. (A
useful rule of thumb for English is about four characters per token; Swahili uses more tokens for the same meaning, as
you saw on Day 7.)

Notice that the **passages** are by far the biggest part: 4 × 700 = 2,800 characters, against a 90-character question.
In RAG, most of what you pay for is the context you send, not the question.

In [ ]:
# Typical sizes, in characters.
SYSTEM_PROMPT_CHARS = 450
PASSAGE_CHARS = 700
PASSAGES = 4
QUESTION_CHARS = 90
ANSWER_CHARS = 350

# About 4 characters per token in English. Swahili needs about 30% more tokens for the same text.
CHARS_PER_TOKEN_EN = 4.0
SWAHILI_EXTRA = 1.3
SHARE_SWAHILI = 0.35               # share of questions asked in Swahili

Turn characters into tokens. The **token factor** is a weighted average: 65% of questions are English (factor 1.0) and
35% are Swahili (factor 1.3), so on average each question uses about 10% more tokens than an all-English one would.

In [ ]:
# Characters sent to the model per question.
chars_in = SYSTEM_PROMPT_CHARS + PASSAGE_CHARS * PASSAGES + QUESTION_CHARS

# The average token "inflation": 65% of questions at 1.0, 35% at 1.3.
token_factor = (1 - SHARE_SWAHILI) * 1.0 + SHARE_SWAHILI * SWAHILI_EXTRA

TOKENS_IN = chars_in / CHARS_PER_TOKEN_EN * token_factor
TOKENS_OUT = ANSWER_CHARS / CHARS_PER_TOKEN_EN * token_factor
print(f"about {TOKENS_IN:.0f} input tokens and {TOKENS_OUT:.0f} output tokens per question")

> **Where we are.** You have the input and output tokens for a typical question, adjusted for the share of Swahili
> questions. Next, write down every other number the costs depend on, in one place.

## 2. The assumptions

Change these to match the Fund. The volumes are the kind of estimate you'd agree with the call-centre and member-service
managers.

**Volume and money.** How many questions a day, how many working days a month, and the exchange rate (cloud bills are in
US dollars; the Fund budgets in shillings).

**Cloud prices.** Two example options: a small, cheap model and a large, expensive one. Output tokens usually cost several
times more than input tokens, because generating is harder than reading. The fixed cloud cost covers the things you pay
for whatever the volume: private networking, monitoring, and a small server to run the API from Lab 2.

In [ ]:
QUESTIONS_PER_DAY = 2_000          # staff + call-centre questions
WORKING_DAYS_PER_MONTH = 22
USD_TO_TZS = 2_600                 # EXAMPLE: use the current rate

# Cloud: US$ per million tokens, input and output. EXAMPLE prices only: look up the current rate for the model and region.
CLOUD_OPTIONS = {
    "Small cloud model (EXAMPLE)": {"in": 0.15, "out": 0.60},
    "Large cloud model (EXAMPLE)": {"in": 2.50, "out": 10.00},
}
CLOUD_FIXED_USD_MONTH = 150        # EXAMPLE: private networking, monitoring, container hosting

**Self-hosting.** The costs of owning a GPU (graphics processing unit) server:

- the purchase price, spread over the years you'll use it (here 4);
- electricity: the server runs all day, every day;
- staff: someone has to install updates, monitor it and fix it. Here a quarter of an engineer's time. This is the cost
  most often forgotten, and it can be the biggest;
- capacity: how many answers per minute one server can produce. A production server running several requests at once
  (**batching**, with serving software such as vLLM) does far more than your laptop did in Lab 2.

In [ ]:
# Self-hosted.
GPU_SERVER_USD = 25_000            # EXAMPLE: one server with a 24-48 GB GPU, from a procurement quotation
YEARS_OF_USE = 4                   # spread the purchase over 4 years
POWER_KW = 1.2                     # EXAMPLE: average power draw
TZS_PER_KWH = 350                  # EXAMPLE: electricity tariff
STAFF_SHARE_FTE = 0.25             # EXAMPLE: a quarter of an engineer's time...
STAFF_COST_TZS_YEAR = 60_000_000   # ...at this yearly cost

# EXAMPLE: a production GPU server with batched serving (e.g. vLLM).
# Your laptop in Lab 2 managed about 20; measure the real server before relying on this.
MAX_ANSWERS_PER_MINUTE = 150

> **Where we are.** The assumptions are set: volume, exchange rate, two cloud price options and the costs of owning a GPU
> server. Each one marked `EXAMPLE` must be replaced before the result is used. Now combine them.

### Theory: why servers are sized for the peak

Questions don't arrive evenly. They bunch up in working hours, and within those hours in the busiest minutes. A server
that can just about keep up with the *average* rate falls behind every peak, and the queue (Lab 2's latency theory) makes
callers wait longer and longer.

So the number of servers is set by the **peak load**, here assumed to be three times the average rate during an 8-hour
working day. That's also why self-hosting is a step function: one server covers volumes up to its peak capacity, and the
next question beyond that needs a whole second server.

## 3. Monthly cost at today's volume

Two functions, one per option, each giving the cost of a month in TZS.

**Cloud:** the cost of one question is (input tokens × input price + output tokens × output price), divided by a million
because prices are per million tokens. Multiply by the number of questions, add the fixed fee, and convert to shillings.

**Self-hosted:** for each server, the purchase price spread over its life (so many months), plus a month of electricity
(kW × 24 hours × 30 days × the tariff). Then add the staff cost once. The number of servers depends on the **peak**: the
questions arrive over about 8 working hours, and the busiest minutes are about three times the average, so the servers
must cope with 3× the average rate. `math.ceil` rounds up (you can't buy half a server), and `max(1, ...)` means you always
need at least one.

In [ ]:
def cloud_month_tzs(questions_per_month, price):
    """Monthly cost of a cloud LLM in TZS: tokens used x price per token, plus fixed fees."""
    cost_per_question_usd = (TOKENS_IN * price["in"] + TOKENS_OUT * price["out"]) / 1_000_000
    tokens_cost_usd = questions_per_month * cost_per_question_usd
    return (tokens_cost_usd + CLOUD_FIXED_USD_MONTH) * USD_TO_TZS


def self_hosted_month_tzs(questions_per_month):
    """Monthly cost of running the model yourself, in TZS: servers (spread over their life), power, staff time."""
    hardware_per_server = GPU_SERVER_USD * USD_TO_TZS / (YEARS_OF_USE * 12)
    power_per_server = POWER_KW * 24 * 30 * TZS_PER_KWH            # running all month
    staff = STAFF_SHARE_FTE * STAFF_COST_TZS_YEAR / 12

    # How many servers? Questions arrive over about 8 working hours, and the busiest minutes
    # are about three times the average, so plan for 3x the average rate.
    average_per_minute = questions_per_month / WORKING_DAYS_PER_MONTH / (8 * 60)
    peak_per_minute = average_per_minute * 3
    servers_needed = max(1, math.ceil(peak_per_minute / MAX_ANSWERS_PER_MINUTE))

    return servers_needed * (hardware_per_server + power_per_server) + staff

Work out all three options at today's volume. `{cost:>14,.0f}` prints the number right-aligned in 14 characters, with
thousands separators and no decimals, so the column lines up.

In [ ]:
questions_per_month = QUESTIONS_PER_DAY * WORKING_DAYS_PER_MONTH

costs = {}
for name, price in CLOUD_OPTIONS.items():
    costs[name] = cloud_month_tzs(questions_per_month, price)
costs["Self-hosted"] = self_hosted_month_tzs(questions_per_month)

for name, cost in costs.items():
    print(f"{name:30s} TZS {cost:>14,.0f} per month")

> **Where we are.** You have each option's monthly cost at today's volume. One volume gives one answer. The committee needs
> to know how the answer changes as volume grows.

## 4. Where does the answer flip?

One volume gives one answer. To see where it changes, work out the monthly cost of each option at 300 different
volumes, from 100 to 100,000 questions a day (`np.linspace` makes 300 evenly spaced values), and draw them. Each loop
fills a list with one monthly cost per volume, in TZS millions. The dashed line marks today's volume.

In [ ]:
volumes = np.linspace(100, 100_000, 300)              # questions per working day

plt.figure(figsize=(9, 4.5))

# The two cloud options.
for name, colour in [("Small cloud model (EXAMPLE)", "#2a78d6"), ("Large cloud model (EXAMPLE)", "#eb6834")]:
    monthly = []
    for v in volumes:
        monthly.append(cloud_month_tzs(v * WORKING_DAYS_PER_MONTH, CLOUD_OPTIONS[name]) / 1e6)
    plt.plot(volumes, monthly, color=colour, label=name)

# Self-hosting.
self_hosted = []
for v in volumes:
    self_hosted.append(self_hosted_month_tzs(v * WORKING_DAYS_PER_MONTH) / 1e6)
plt.plot(volumes, self_hosted, color="black", lw=2, label="Self-hosted")

# Mark today's volume.
plt.axvline(QUESTIONS_PER_DAY, color="grey", ls="--")
plt.text(QUESTIONS_PER_DAY * 1.05, plt.ylim()[1] * 0.9, "today", color="grey")
plt.xlabel("questions per working day")
plt.ylabel("TZS millions per month")
plt.title("Monthly cost by volume (EXAMPLE prices: replace before use)", loc="left")
plt.legend()
plt.show()

Find the break-even point for each cloud option: go through the volumes from smallest to largest, and return the first
one at which the cloud option costs more than self-hosting. If that never happens in the range, return `None`.

In [ ]:
def break_even(price):
    """The first volume (questions per day) at which the cloud option costs more than self-hosting, or None."""
    for v in volumes:
        cloud = cloud_month_tzs(v * WORKING_DAYS_PER_MONTH, price)
        own = self_hosted_month_tzs(v * WORKING_DAYS_PER_MONTH)
        if cloud > own:
            return int(v)
    return None


for name, price in CLOUD_OPTIONS.items():
    volume = break_even(price)
    if volume is None:
        print(name + ": stays cheaper than self-hosting across the whole range shown")
    else:
        print(f"{name}: self-hosting becomes cheaper above about {volume:,} questions per working day")

Read the chart the way the steering committee will:

- A **small** cloud model can stay cheaper than owning a GPU (graphics processing unit) server even at high volumes, once
  staff time is counted. Whether its *answers* are good enough is a separate question, for your Day 7-style tests.
- A **large** cloud model gets expensive as volume grows, and the bill is in dollars. Somewhere along the volume axis,
  owning the hardware wins.
- Self-hosting is a **step** function: cheap per question once the server exists, then another step when you need a second one.

And remember what the chart leaves out: answer quality (a large cloud model may answer far better), data protection (only
self-hosting keeps prompts in the country), and risk (price changes, exchange rates, an internet outage).

> **Where we are.** The chart shows each option's cost across volumes, and the break-even points show where owning the
> hardware becomes cheaper than each cloud option. You've also seen what the chart leaves out: answer quality, data
> protection and risk.

## 5. Your turn

1. Replace the `EXAMPLE` values with current figures you can find or estimate. Does the conclusion change?
2. Scenario: the Fund opens the assistant to **members** on WhatsApp, and volume rises to 30,000 questions a day. Which option
   is cheapest now? What else changes besides cost?
3. Add a row to the side-by-side table in the guide with the numbers from this worksheet.

In [ ]:
# Your code here

## Looking back, looking ahead

**What you did in this worksheet.** You estimated the tokens in one RAG question, set out every cost assumption in one
place, worked out a month's cost for two cloud options and for self-hosting (sized for peak load), and found the volumes
at which the cheapest option changes.

**What you did today.** Day 9 built the handbook assistant properly: RAG with current documents, citations, a threshold
and a guardrail (Lab 1), an internal API with keys, rate limits and safe logging (Lab 2), and the cost and data case for
where to run it (Lab 3).

**The ideas to carry forward.**

- In RAG, most of the tokens you pay for are the passages you send, not the question.
- Fixed versus variable cost decides the answer; the break-even point is the useful output.
- Size servers for the peak; count staff time.
- Cost is one column of the decision; quality, data protection and risk are the others.

**Tomorrow (Day 10).** Deploy, monitor and govern: the claims risk model from Day 2 as a containerised service, drift and
performance monitoring on six months of live scoring, and the governance pack.